In [0]:
# Databricks notebook source
import time
import mlflow
import pandas as pd
import numpy as np

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

# 1. Cargar tablas generadas
df_silver = spark.table(f"{catalog}.{schema}.silver_cement_quality")
df_gold = spark.table(f"{catalog}.{schema}.gold_cement_features")
df_preds = spark.table(f"{catalog}.{schema}.ml_test_predictions").toPandas()
df_champ = df_preds[df_preds["is_champion"] == True]

# 2. Métricas de error y residuales (formato largo: una fila por muestra, target y modelo)
df_r1 = df_champ[df_champ["target"] == "Strength1"]
df_r7 = df_champ[df_champ["target"] == "Strength7"]
df_r28 = df_champ[df_champ["target"] == "Strength28"]

mape_1 = np.mean(np.abs(df_r1["y_true"] - df_r1["y_pred"]) / df_r1["y_true"]) * 100
mape_7 = np.mean(np.abs(df_r7["y_true"] - df_r7["y_pred"]) / df_r7["y_true"]) * 100
mape_28 = np.mean(np.abs(df_r28["y_true"] - df_r28["y_pred"]) / df_r28["y_true"]) * 100

mean_residual_28 = np.mean(df_r28["y_true"] - df_r28["y_pred"])
bias_type = "Conservador (Subestimacion segura)" if mean_residual_28 > 0 else "Agresivo"

# 3. Monotonía física
pivot = df_champ.pivot_table(index="sample_id", columns="target", values="y_pred")
violations = int(((pivot["Strength1"] > pivot["Strength7"]) | (pivot["Strength7"] > pivot["Strength28"])).sum())
n_samples = len(pivot)

# 4. Benchmark de inferencia con el modelo campeón
model_uri = f"models:/{catalog}.{schema}.cement_strength_predictor@champion"
loaded_model = mlflow.pyfunc.load_model(model_uri)

# La firma del modelo espera las columnas de features de Gold (no las de la tabla de predicciones)
input_cols = [c.name for c in loaded_model.metadata.get_input_schema().inputs]
sample_batch = df_gold.select(*input_cols).limit(100).toPandas()
t0 = time.time()
_ = loaded_model.predict(sample_batch)
latency_ms = ((time.time() - t0) / 100) * 1000

# 5. Reporte integral estructurado
report = f"""
================================================================================
                    REPORTE DE AUDITORIA Y GOBERNANZA - GCC
================================================================================
ESTADO DEL PIPELINE: TERMINATED SUCCESS
CATALOGO / ESQUEMA:  {catalog}.{schema}
MODELO CAMPEON:      ExtraTreesRegressor (Registrado bajo alias @champion)

--------------------------------------------------------------------------------
1. LINAJE Y CALIDAD DE DATOS (DATA HYGIENE)
--------------------------------------------------------------------------------
* Registros procesados (Silver): {df_silver.count():,} filas
* Registros feature store (Gold): {df_gold.count():,} filas
* Multicolinealidad (VIF):        Mineral_Phase_C removida (VIF > 10.0 resuelto)
* Encoding seguro:               pt_other excluido (Dummy variable trap evitado)

--------------------------------------------------------------------------------
2. METRICAS DEL MODELO CAMPEON (CONJUNTO DE PRUEBA)
--------------------------------------------------------------------------------
* MAPE 1 Dia (Fuerza Temprana):   {mape_1:.2f}%
* MAPE 7 Dias (Transicion):       {mape_7:.2f}%
* MAPE 28 Dias (Fuerza Final):    {mape_28:.2f}%
* Monotonia Fisica (R1<=R7<=R28): {'CUMPLE' if violations == 0 else 'FALLA'} ({violations} violaciones en {n_samples} muestras)

--------------------------------------------------------------------------------
3. EVALUACION DE RIESGO INDUSTRIAL Y RESIDUALES
--------------------------------------------------------------------------------
* Media del residual (Dia 28):   {mean_residual_28:+.2f} psi
* Perfil de seguridad industrial: {bias_type}
* Intervalos Conformal (90% CI):  Activos en inferencia PyFunc

--------------------------------------------------------------------------------
4. EFICIENCIA OPERATIVA EN PRODUCCION
--------------------------------------------------------------------------------
* Latencia promedio de inferencia: {latency_ms:.2f} ms / muestra
* Disponibilidad en Unity Catalog: Activa para Batch / Streaming / BI Dashboards
================================================================================
"""

print(report)
dbutils.notebook.exit(report)